# Lesson 05 Lab — Diagnosing FP16 Overflow and Gradient Scaling Failures

**Puzzle:** When loss becomes NaN, how do we distinguish forward overflow, backward overflow, and gradient underflow?

The saved outputs were generated by executing every code cell on the recorded RTX 5090. Run all cells to regenerate the evidence on your own CUDA GPU.

## 0. Predict before running

Write down: (1) the expected direction, (2) the mechanism, (3) the observation that would reverse your prediction, and (4) the evidence level required for the claim.

## 1. Theory — objects and data flow

Diagnose four checkpoints: forward outputs, scaled loss/gradients, unscaled gradients, and post-step parameters. A final NaN has already discarded the location of the first failure.

### Core mechanism

FP16 normal values end near `6.55e4`; very small values enter a sparse subnormal region and can become zero. Loss scaling shifts gradient magnitudes upward during storage, but unscaling must happen before clipping and parameter updates.

In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "05-fp16-overflow"
device = require_cuda()
torch.manual_seed(2026 + 5)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 2. Connect theory to the experiment

### Engineering trade-off

An aggressive scale protects small gradients but increases overflow risk. A conservative scale avoids Inf yet may leave many gradients at zero, so the useful interval is workload-dependent.

### What this code tests

The CUDA sweep crosses both tiny and large magnitudes at several scales and records zero and Inf fractions, making the failure stage observable.

**Experiment:** Sweep synthetic gradient magnitudes and loss scales in FP16 on CUDA, counting finite, infinite, and zero gradient values.

**Declared evidence label:** `pytorch-gpu`. Check that the shapes, controlled variables, and units match the theoretical question before executing.

In [2]:
rows = []
for magnitude in (1e-8, 1e-5, 1.0, 1e3):
    for scale in (1.0, 256.0, 65536.0):
        p = torch.ones(4096, device=device, dtype=torch.float16, requires_grad=True)
        loss = (p.float() * magnitude).sum() * scale
        loss.backward(); g = p.grad
        rows.append({"magnitude": magnitude, "loss_scale": scale,
                     "zero_fraction": round((g == 0).float().mean().item(), 6),
                     "inf_fraction": round(torch.isinf(g).float().mean().item(), 6),
                     "finite_fraction": round(torch.isfinite(g).float().mean().item(), 6)})
forward_overflow = torch.isinf(torch.tensor([1e5], device=device).half()).item()
result = base_result(5, "pytorch-gpu"); result.update({"gradient_sweep": rows,
    "forward_overflow_at_1e5": bool(forward_overflow),
    "conclusion": "Scaling changed gradient representability but could not repair an FP16 value that had already overflowed."})


## 3. Inspect the evidence

The useful evidence is the first stage where finiteness changes. A final NaN without intermediate checks is not a diagnosis.

### Acceptance and rollback gate

Log finite/Inf/zero fractions and the current scale. If the forward pass is already non-finite, change the operation or dtype; if only scaled gradients overflow, adjust scale policy.

In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "conclusion": "Scaling changed gradient representability but could not repair an FP16 value that had already overflowed.",
  "environment": {
    "compute_capability": "12.0",
    "cuda_runtime": "13.0",
    "gpu": "NVIDIA GeForce RTX 5090",
    "gpu_memory_gib": 31.358,
    "python": "3.12.13",
    "torch": "2.12.0"
  },
  "evidence_label": "pytorch-gpu",
  "executed_at_utc": "2026-08-07T14:45:17+00:00",
  "forward_overflow_at_1e5": true,
  "gradient_sweep": [
    {
      "finite_fraction": 1.0,
      "inf_fraction": 0.0,
      "loss_scale": 1.0,
      "magnitude": 1e-08,
      "zero_fraction": 1.0
    },
    {
      "finite_fraction": 1.0,
      "inf_fraction": 0.0,
      "loss_scale": 256.0,
      "magnitude": 1e-08,
      "zero_fraction": 0.0
    },
    {
      "finite_fraction": 1.0,
      "inf_fraction": 0.0,
      "loss_scale": 65536.0,
      "magnitude": 1e-08,
      "zero_fraction": 0.0
    },
    {
      "finite_fraction": 1.0,
      "inf_fraction": 0.0,
      "loss_scale

## 4. Explain the result

Place finiteness and zero-rate probes at forward outputs, scaled gradients, unscaled gradients, and parameters before changing the scaler policy.

Relate the measured fields back to the mechanism above. Treat the checked-in result as one hardware/software observation, not a universal ranking. The complete derivation, evidence boundary, and primary references are in [`README.md`](README.md).